### Byte Pair Encoding — tiktoken example

In [ ]:
import tiktoken

In [ ]:
# with open("the-verdict.txt", "r") as f:
#     text = f.read()

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
sample = "Hello world <|endoftext|> "
tokenizer.encode(sample, allowed_special={"<|endoftext|>"})

In [ ]:
tokenizer.decode([15496, 995, 220, 50256, 220])

### Byte Pair Encoding — from-scratch implementation

In [ ]:
text = """The Internet is a global system that connects computers and devices worldwide, enabling communication, information sharing, and access to digital services.

Connects people and devices globally
Enables communication, learning, and business
Supports digital services and cloud platforms
Operates using standard communication protocols
Forms the backbone of modern digital life """

In [ ]:
encoded_bytes = list(text.encode(encoding="utf-8"))

In [ ]:
encoded_bytes

In [ ]:
char_freq = {char: encoded_bytes.count(char) for char in encoded_bytes}
sorted(char_freq.items(), key=lambda x: x[1], reverse=True)

In [ ]:
pairs = list(zip(encoded_bytes, encoded_bytes[1:]))

In [ ]:
pair_count = {p: pairs.count(p) for p in pairs}

In [ ]:
pair_count

In [ ]:
sorted_pair = dict(sorted(pair_count.items(), key=lambda x: x[1], reverse=True))
print("The most occuring pairs are")
sorted_pair

In [ ]:
top_pair = max(sorted_pair, key=sorted_pair.get)
top_pair

In [ ]:
def merge_id(ids: list, pair_seq: tuple, new_id: int) -> list:

    """Replace every consecutive occurrence of pair_seq in ids with new_id.

        ids:Current list of tokens 
        pair_seq:top sequence we merge.
        new_id: new id to replace top sequence with
    """
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair_seq[0] and ids[i + 1] == pair_seq[1]:
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

In [ ]:
new_ids = merge_id(encoded_bytes, top_pair, max(encoded_bytes) + 1)
new_ids

In [ ]:
def encode_seq(text: str, epochs: int):
    """BPE encode text by merging the most frequent pair `epochs` times.
    text: input text to encode
    epochs: how many times to merge
    """
    ids = list(text.encode(encoding="utf-8"))
    merges = {} 

    for i in range(epochs):
        pairs = list(zip(ids, ids[1:]))
        pair_count = {p: pairs.count(p) for p in pairs}
        sorted_pairs = dict(sorted(pair_count.items(), key=lambda x: x[1], reverse=True))
        top_pair = max(sorted_pairs, key=sorted_pairs.get)
        merge_by = max(ids) + 1
        ids = merge_id(ids, top_pair, merge_by)
        print(f"Merging {top_pair} -> {merge_by}")
        merges[merge_by] = [top_pair[0], top_pair[1]]

    return ids, merges

In [ ]:
def recursive_check(merges: dict, token_id: int) -> list:
    """ Recursively expand a merged token id back to its raw byte ids
    by looking at merges dictionary.
    """
    
    if token_id not in merges:
        return [token_id]

    result = []
    for child in merges[token_id]:
        result.extend(recursive_check(merges, child))
    return result

In [ ]:
def decode_seq(ids: list, merges: dict) -> str:
    """
    Decode the ids back to string
    """

    result = ""
    for token_id in ids:
        raw_bytes = recursive_check(merges, token_id) ## check if its merged 
        result += "".join(bytes([b]).decode("utf-8", errors="replace") for b in raw_bytes)
    return result

In [ ]:
sample_text = text[:55]
encoded, merges = encode_seq(sample_text, 4)
decoded = decode_seq(encoded, merges)
print("Original : ", repr(sample_text))
print("Decoded  : ", repr(decoded))
print("Match    : ", sample_text == decoded)